# Laboratorijske vaje 6
## 1. Križna korelacija

$r_{xy}(l)=\sum_{n=-\infty}^{\infty}x(n)y(n-l)$, $l=0,\pm 1,\pm 2$ 
    
Imamo ooenostavljen radarski sistem  ($F_s=1 MHz$). Paket je najmanjša celota signala, ki se oddaja (v tem primeru zaporedje naključnih števil). Prejeti signal (*recx*) je zamaknjen in moten oddajan signal (*sentx*).

V datoteki radar.mat se nahajata 2 para oddanih in prejetih signalov ("sent1", "rec1", "sent2", "rec2"), ter "paket", ki vsebuje zaporedje naključnih števil, in z ničlami dopolnjen "paket0".
Določite oddaljenost objektov v obeh primerih ob pogoju, da je do odboja sploh prišlo. Namig: izračunajte križne korelacije (kovariance) med zaporedji: ```sgn.correlate(paket0, sentx)``` in ```sgn.correlate(paket0, recx)``` in ugotovite razdaljo med maksimumi – ta pomeni zakasnitev paketa v številu vzorcev.

Primer računanja (avto, križne) korelacije:
```Python
korelacija = sgn.correlate(x, y, mode='full')
zakasnitve = sgn.correlation_lags(len(x), len(y))
```

Dodatni vir: 
- http://en.wikipedia.org/wiki/Radar

In [ ]:
%matplotlib ipympl
import numpy as np
from scipy.io import loadmat
import matplotlib.pyplot as plt
import scipy.signal as sgn

radar = loadmat('radar.mat')
paket = radar.get('paket').flatten()


## 2. Ugotavljanje in odpravljanje odmeva
Imamo posnetek nekega govora brez in z odmevom ($F_s=22050Hz$): danes_je_lep_dan.wav in danes_je_lep_dan_odmev.wav.
S pomočjo križne korelacije poskušajte ugotoviti locirati odmev v posnetku z odmevom in določite oba parametra odmeva: zakasnitev in njegovo relativno amplitudo (136,054 ms, ~0,6).

Za odpravljanje odmeva moramo določiti diferenčno enačbo t.i. inverznega sistema (torej sistema, ki bo signal z odmevom povrnil v prvotni signal brez odmeva). Za inverzni sistem velja:

$H_I(z) = H^{-1}(z)$

Najprej v celoti določite diferenčno enačbo sistema za dodajanje odmeva na tej osnovi: 

$y(n)=x(n)+Ax(n-D)$, kjer je $D$ zakasnitev odmeva in $A$ njegova relativna amplituda.

Nato še določite diferenčno enačbo inverznega sistema (Namig: diferenčno enačbo inverznega sistema lahko dobimo tudi bolj elegantno – in sicer zamenjamo vhodni in izhodni signal ter izraz preoblikujemo v diferenčno enačbo inverznega sistema).

Izvedite inverzni sistem na posnetku z odmevom in preverite rezultat.

In [ ]:
import numpy as np
from scipy.io import wavfile
import scipy.signal as sgn
import sounddevice as sd


# Praktični primeri

## P1. Generator DTMF telefonskega signala (tonsko izbiranje)

Pritisk na vsako tipko na telefonski številčnici aktivira generiranje signala, ki je sestavljen iz dveh frekvenčnih komponent. Ena komponenta je iz skupine spodnjih frekvenc (697 - 941) in druga iz skupine zgornjih frekvenc (1209 - 1477). 

![dtmf](dtmf_freq.svg)

Vrednosti vseh frekvenc so natančno preračunane, da bi bilo število napak pri razpoznavanju generiranih tonov čimmanjše in sicer:
- Nobena od frekvenc ni 2x višja od katerikoli druge.
- Prav tako ni mnogokratnik od katerikoli vsote ali razlike dveh frekvenc.

Za generiranje signalov imamo več možnosti. Najenostavnejša med njimi je direktno generiranje vrednosti sin/cos.
    
a. Napišite preprost program, ki bo generiral ustrezni signal za tonsko izbiranje izbrane tipke in slušno preverite rezultat. Pri tem uporabite funkcijo dtmf_2f (datoteka *dmtf.py*), ki vam za dano tipko vrne dvojico (tuple) frekvenc obeh tonov, ki jih potrebujete za DTMF zvok.

Tvorba tonov z direktnim računanjem vrednosti sin/cos, lahko pri implementaciji na nižjem nivoju povzroči težave, ker običajno nimamo na voljo funkcije za izračun sin/cos. V tem primeru bi si lahko pomagali tudi z vnaprej izračunanimi tabelami. Obstaja pa bolj elegantna rešitev s pomočjo sicer nestabilnega rekurzivnega sistema (s poli na enotski krožnici v Z-ravnini), ki ga imenujemo tudi oscilator (s pravilnim zbujanjem ga v tem primeru naredimo stabilnega):

$y(n)=2\cos(W)y(n-1)-y(n-2)$, $y(-1)=1$, $W=2\pi \frac{F}{F_s}$

b. Tvorite DTMF zvok podobno kot v podnalogi a), vendar z uporabo oscilatorja. Preverite rezultat.

**Pomoč:** Rekurzivni sistem lahko na kratko izračunate s funkcijo ```sgn.lfilter```, ki kot vhodni parameter sprejme koeficiente a in b differenčne enačbe. S funkcijo ```sgn.lfilteric``` kreirate začetne pogoje. 

```Python
# Koeficienti diferenčne enačbe  
a = []
b = []
zacetno_stanje = sgn.lfiltic(b, a, y, x) # y in x sta arraya preteklih vrednosti vhodnih in izhodnih vzorcev
signal, zacetno_stanje = sgn.lfilter(b, a, vhodni_signal, zi=zacetno_stanje)
```

Sistem lahko implementirate tudi s pomočjo for zanke.

In [ ]:
%matplotlib ipympl
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as sgn
import sounddevice as sd
from dtmf import dtmf_2f


## P2. Zvočni iskalnik s pomočjo križne korelacije

S pomočjo križne korelacije lahko poiščemo pojavitev nekega znanega dela signala v nekem neznanem signalu. Uporabite posnetek *Generated_Rain_Scene_8000.wav* v katerem bomo iskali posnetek kukavice. Posnetek zvočnega detajla, ki ga iščemo, se nahaja v datoteki *Cuckoo_mono_8000_detajl.wav*.

Naloge:
- S križno korelacijo ugotovite kje v posnetku se pojavi zvok kukavice. 
- S pomočjo programa Audacity (ali kakega drugega avdio analizatorja/predvajalnika) preverite, ali se zvok kukavice res nahaja na prikazanih pozicijah v zvočni sceni.

In [ ]:
%matplotlib ipympl
import numpy as np
from scipy.io import wavfile
import matplotlib.pyplot as plt
import scipy.signal as sgn
import sounddevice as sd


# Neobvezno, a zanimivo

## N1. Zaznavanje normalnega in nenormalnega srčnega utripa v elektrokardiogramu (ECG)

Na voljo zapis ECG z imenom *106.mat*, ki vsebuje dva hkrati posneta ECG signala. Na voljo je tudi datoteka z ustrezno referenčno anotacijo pojavov srčnih utripov z imenom *106NV.txt*.

- Prvi stolpec v datoteki *106NV.txt* predstavlja indeks pojavov srčnega utripa.
- Drugi stolpec predstavlja tip srčnega utripa.

Tipi srčnih utripov so označeni:
- **N** – normalen srčni utrip  
- **V** – nenormalen srčni utrip  

Ta zapis je del baze podatkov [MIT-BIH Arrhythmia Database](https://www.physionet.org/physiobank/database/mitdb/)

Signal je vzorčen s frekvenco 360 vzorcev na sekundo.

**Naloge:**
- Na začetku ECG zapisa poiščite nekaj primerov normalnih in nenormalnih srčnih utripov s pomočjo referenčne anotacijske datoteke ter ustvarite dva referenčna vzorca srčnega utripa.
- Z uporabo ustvarjenih vzorcev poiščite pojave normalnih in nenormalnih srčnih utripov v zapisu. Za iskanje pojavov srčnih utripov izvedite korelacijo med referenčnima vzorcema in ECG signalom. Svoje rezultate lahko primerjate s skupno statistiko srčnih utripov na naslednji spletni strani: https://www.physionet.org/physiobank/database/html/mitdbdir/records.htm
- Zgornje naloge izvedite dvakrat (za oba ECG signala v zapisu).
- Ali obstaja razlika v skupnem številu normalnih in nenormalnih srčnih utripov med obema signaloma?